In [1]:
import json
import re
import sqlite3
import os

In [2]:
SPIDER_DB_PATH = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database" 

def execute_query(db_id, sql):
    db_path = os.path.join(SPIDER_DB_PATH, db_id, f"{db_id}.sqlite")
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        results = set(cursor.fetchall())
        conn.close()
        return results
    except Exception as e:
        return None  # query inválida

def execution_accuracy(gold, pred, db_id):
    gold_results = execute_query(db_id, gold)
    pred_results = execute_query(db_id, pred)
    if pred_results is None:
        return False
    return gold_results == pred_results

def normalize_sql(sql: str) -> str:
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)         
    sql = sql.rstrip(';')                 
    return sql

def exact_match(gold: str, pred: str) -> bool:
    return normalize_sql(gold) == normalize_sql(pred)

In [3]:
with open("predictions_r64q4a128_2.json") as f:
    predictions = json.load(f)

In [4]:
em_scores = [exact_match(p["gold"], p["predicted"]) for p in predictions]
em = sum(em_scores) / len(em_scores)
print(f"Exact Match: {em:.4f} ({sum(em_scores)}/{len(em_scores)})")

Exact Match: 0.4632 (479/1034)


In [5]:
ex_scores = [
    execution_accuracy(p["gold"], p["predicted"], p["db_id"])
    for p in predictions
]
ex = sum(ex_scores) / len(ex_scores)
print(f"Execution Accuracy: {ex:.4f} ({sum(ex_scores)}/{len(ex_scores)})")

Execution Accuracy: 0.7476 (773/1034)
